# 01 - Build Dataset
Construct the bed-foot removal dataset from PartNet raw mesh data.

**Run cells one by one.** Cell 3 tests on a single sample first to catch problems before the full build.

In [1]:
import sys, os, gc, json
import numpy as np

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Increase recursion limit for deep PartNet JSON trees
sys.setrecursionlimit(5000)

from src.config import load_config, ensure_dirs
print('Modules loaded OK')

Modules loaded OK


In [2]:
# Load configuration
cfg = load_config(os.path.join(PROJECT_ROOT, 'configs', 'bed_leg.yaml'))
ensure_dirs(cfg)

PARTNET_ROOT = cfg['paths']['partnet_root']
OUTPUT_DIR   = cfg['paths']['raw_data_dir']

print(f'PartNet root : {PARTNET_ROOT}')
print(f'Output dir   : {OUTPUT_DIR}')
print(f'Category     : {cfg["dataset"]["category"]}')
print(f'Remove part  : {cfg["dataset"]["semantic_label"]}')
print(f'Total samples: {cfg["dataset"]["total_samples"]}')

# Sanity check PartNet path
if not os.path.isdir(PARTNET_ROOT):
    raise FileNotFoundError(
        f'PartNet directory not found: {PARTNET_ROOT}\n'
        f'Edit configs/default.yaml -> paths.partnet_root'
    )

subdirs = [d for d in os.listdir(PARTNET_ROOT)
           if os.path.isdir(os.path.join(PARTNET_ROOT, d))]
print(f'\nPartNet has {len(subdirs)} subdirectories')
print(f'First 5: {subdirs[:5]}')

PartNet root : E:/datasets/partnet
Output dir   : D:/MyJupyter/Works/3DPART_v2/data/raw/bed_leg_removal
Category     : Bed
Remove part  : leg
Total samples: 50

PartNet has 32537 subdirectories
First 5: ['1000', '10000', '10003', '10005', '10007']


In [3]:
# ============================================================
# STEP 1: Quick scan to find StorageFurniture samples with "door" parts
#         (no mesh loading, only reads JSON - safe & fast)
# ============================================================
from src.data.dataset_builder import DatasetBuilder

builder = DatasetBuilder(
    partnet_root=PARTNET_ROOT,
    output_dir=OUTPUT_DIR,
    category=cfg['dataset']['category'],
    semantic_label=cfg['dataset']['semantic_label'],
    total_samples=cfg['dataset']['total_samples'],
    random_seed=cfg['dataset']['random_seed'],
)

print('Scanning for candidate samples (JSON only, no mesh loading)...')
candidates = builder._find_partnet_samples()
print(f'Found {len(candidates)} candidate {cfg["dataset"]["category"]} samples with {cfg["dataset"]["semantic_label"]} parts')

if len(candidates) > 0:
    print(f'First 5 candidate IDs: {[os.path.basename(c) for c in candidates[:5]]}')
else:
    print('ERROR: No candidates found. Check PartNet directory structure.')

Scanning for candidate samples (JSON only, no mesh loading)...


2026-04-15 19:59:50 [INFO] DatasetBuilder: Scan complete: 163 valid, 32289 skipped                                     


Found 163 candidate Bed samples with leg parts
First 5 candidate IDs: ['10027', '10033', '10039', '10049', '10161']


In [4]:
# ============================================================
# STEP 2: Test on ONE sample before full build
#         If this cell crashes, the issue is in mesh loading
# ============================================================
from src.io.mesh_io import load_mesh_lightweight, merge_arrays, save_mesh
from pathlib import Path

test_dir = candidates[0]
test_id = os.path.basename(test_dir)
print(f'Testing on sample: {test_id}')

# Parse annotation
anno_file = os.path.join(test_dir, 'result_after_merging.json')
if not os.path.exists(anno_file):
    anno_file = os.path.join(test_dir, 'result.json')

with open(anno_file, 'r', encoding='utf-8') as f:
    anno = json.load(f)

part_groups = builder._find_part_objs(anno, cfg['dataset']['semantic_label'])
all_obj_ids = builder._collect_all_obj_ids(anno)
print(f'  Total OBJ IDs: {len(all_obj_ids)}')
print(f'  {cfg["dataset"]["semantic_label"].capitalize()} part groups: {len(part_groups)}')
print(f'  First group has {len(part_groups[0])} OBJs: {part_groups[0][:5]}')

# Test loading a single OBJ
objs_dir = os.path.join(test_dir, 'objs')
test_obj = os.path.join(objs_dir, f'{all_obj_ids[0]}.obj')
print(f'\n  Test loading: {test_obj}')
md = load_mesh_lightweight(test_obj)
if md is not None:
    print(f'  OK: {md["vertices"].shape[0]} vertices, {md["faces"].shape[0]} faces')
    del md
else:
    print('  FAILED to load. Check OBJ file format.')

gc.collect()
print('\n  Single OBJ test passed!')

Testing on sample: 10027
  Total OBJ IDs: 361
  Leg part groups: 2
  First group has 16 OBJs: ['new-27', 'new-26', 'new-29', 'new-24', 'new-28']

  Test loading: E:\datasets\partnet\10027\objs\new-31.obj
  OK: 5071 vertices, 9872 faces

  Single OBJ test passed!


In [5]:
# ============================================================
# STEP 3: Process the test sample end-to-end
# ============================================================
import random
random.seed(42)

remove_group = random.choice(part_groups)
remove_set = set(remove_group)

removed_paths = []
remaining_paths = []
for obj_id in all_obj_ids:
    obj_file = os.path.join(objs_dir, f'{obj_id}.obj')
    if not os.path.exists(obj_file):
        continue
    if obj_id in remove_set:
        removed_paths.append(obj_file)
    else:
        remaining_paths.append(obj_file)

print(f'  Removed part OBJs: {len(removed_paths)}')
print(f'  Remaining OBJs:    {len(remaining_paths)}')

# Load removed part
removed_dicts = [load_mesh_lightweight(p) for p in removed_paths]
removed_dicts = [d for d in removed_dicts if d is not None]
removed_mesh = merge_arrays(removed_dicts)
print(f'  Removed mesh: {len(removed_mesh.vertices)} verts, {len(removed_mesh.faces)} faces')
del removed_dicts

# Load remaining parts
remaining_dicts = [load_mesh_lightweight(p) for p in remaining_paths]
remaining_dicts = [d for d in remaining_dicts if d is not None]
damaged_mesh = merge_arrays(remaining_dicts)
print(f'  Damaged mesh: {len(damaged_mesh.vertices)} verts, {len(damaged_mesh.faces)} faces')
del remaining_dicts

# Save test output
test_out = os.path.join(OUTPUT_DIR, '_test_sample')
os.makedirs(test_out, exist_ok=True)
save_mesh(damaged_mesh, os.path.join(test_out, 'damaged.obj'))
save_mesh(removed_mesh, os.path.join(test_out, 'removed_part.obj'))

del damaged_mesh, removed_mesh
gc.collect()

print(f'\n  Single sample test PASSED! Files saved to {test_out}')
print('  Safe to proceed with full build.')

  Removed part OBJs: 48
  Remaining OBJs:    313
  Removed mesh: 212028 verts, 421008 faces
  Damaged mesh: 393772 verts, 779177 faces

  Single sample test PASSED! Files saved to D:/MyJupyter/Works/3DPART_v2/data/raw/bed_leg_removal\_test_sample
  Safe to proceed with full build.


In [6]:
# ============================================================
# STEP 4: Full dataset build (uses memory-safe builder)
# ============================================================

# Clean up test
import shutil
test_out = os.path.join(OUTPUT_DIR, '_test_sample')
if os.path.exists(test_out):
    shutil.rmtree(test_out)

gc.collect()

# Rebuild the builder with fresh state
builder = DatasetBuilder(
    partnet_root=PARTNET_ROOT,
    output_dir=OUTPUT_DIR,
    category=cfg['dataset']['category'],
    semantic_label=cfg['dataset']['semantic_label'],
    total_samples=cfg['dataset']['total_samples'],
    random_seed=cfg['dataset']['random_seed'],
)

output_path = builder.build()
print(f'\nDataset saved to: {output_path}')

2026-04-15 19:59:56 [INFO] DatasetBuilder: Scanning PartNet for Bed with 'leg' parts...
2026-04-15 20:00:01 [INFO] DatasetBuilder: Scan complete: 163 valid, 32289 skipped                                     
2026-04-15 20:00:01 [INFO] DatasetBuilder: Selected 50 samples for dataset
Building dataset: 100%|████████████████████████████████████████████████████████████████| 50/50 [02:20<00:00,  2.80s/it]
2026-04-15 20:02:21 [INFO] DatasetBuilder: Dataset built: 50/50 samples saved to D:\MyJupyter\Works\3DPART_v2\data\raw\bed_leg_removal



Dataset saved to: D:\MyJupyter\Works\3DPART_v2\data\raw\bed_leg_removal


In [7]:
# Verify dataset
from src.data.dataset_index import DatasetIndex

index = DatasetIndex(OUTPUT_DIR)
print(f'Total samples in dataset: {len(index)}')
print(f'\nFirst 10 sample IDs:')
for sid in index.sample_ids[:10]:
    meta_path = os.path.join(OUTPUT_DIR, sid, 'meta.json')
    if os.path.exists(meta_path):
        with open(meta_path) as f:
            meta = json.load(f)
        print(f'  {sid}: {meta["mesh_before"]["n_vertices"]} -> '
              f'{meta["mesh_after"]["n_vertices"]} verts '
              f'(removed {len(meta["removed_obj_files"])} OBJs)')

Total samples in dataset: 50

First 10 sample IDs:
  11341: 38208 -> 32993 verts (removed 27 OBJs)
  12467: 47029 -> 44679 verts (removed 20 OBJs)
  12474: 43927 -> 32317 verts (removed 24 OBJs)
  10995: 524460 -> 457842 verts (removed 24 OBJs)
  12802: 177982 -> 153544 verts (removed 54 OBJs)
  12770: 259960 -> 84500 verts (removed 806 OBJs)
  12274: 566064 -> 565920 verts (removed 18 OBJs)
  10312: 59117 -> 48941 verts (removed 36 OBJs)
  12787: 119067 -> 112839 verts (removed 24 OBJs)
  11593: 46073 -> 31067 verts (removed 42 OBJs)


In [8]:
# Create train/val/test splits
splits = index.save_splits(cfg['paths']['splits_dir'])
for name, ids in splits.items():
    print(f'  {name}: {len(ids)} samples')
print('\nDone! Proceed to notebook 02.')

  train: 35 samples
  val: 7 samples
  test: 8 samples

Done! Proceed to notebook 02.


In [9]:
import os
import json
from collections import Counter

PARTNET_ROOT = cfg['paths']['partnet_root']
TARGET_CATEGORY = "StorageFurniture"

def collect_names(node, counter):
    if isinstance(node, dict):
        name = node.get("name", None)
        if isinstance(name, str) and name.strip():
            counter[name.strip().lower()] += 1
        for v in node.values():
            collect_names(v, counter)
    elif isinstance(node, list):
        for x in node:
            collect_names(x, counter)

name_counter = Counter()
matched_ids = []

for sid in os.listdir(PARTNET_ROOT):
    sample_dir = os.path.join(PARTNET_ROOT, sid)
    if not os.path.isdir(sample_dir):
        continue

    meta_path = os.path.join(sample_dir, "meta.json")
    anno_path = os.path.join(sample_dir, "result_after_merging.json")

    if not os.path.isfile(meta_path) or not os.path.isfile(anno_path):
        continue

    try:
        with open(meta_path, "r", encoding="utf-8") as f:
            meta = json.load(f)
        category = str(meta.get("model_cat", "")).strip()
        if category != TARGET_CATEGORY:
            continue

        with open(anno_path, "r", encoding="utf-8") as f:
            anno = json.load(f)

        matched_ids.append(sid)
        collect_names(anno, name_counter)

    except Exception:
        continue

print(f"{TARGET_CATEGORY} samples found: {len(matched_ids)}")
print("\nTop 80 part names:")
for name, cnt in name_counter.most_common(80):
    print(f"{name:30s} {cnt}")

StorageFurniture samples found: 2639

Top 80 part names:
frame_horizontal_bar           6373
handle                         6178
shelf                          5939
vertical_side_panel            5599
drawer_front                   4158
drawer                         4143
drawer_box                     4142
cabinet_door                   3284
cabinet_door_surface           3144
frame_vertical_bar             2832
bottom_panel                   2751
storage_furniture              2639
cabinet_frame                  2611
cabinet                        2599
drawer_side                    2590
base_side_panel                2542
back_panel                     2476
foot                           2152
vertical_divider_panel         2135
top_panel                      2113
cabinet_base                   1266
drawer_back                    1262
drawer_bottom                  1255
panel_base                     684
foot_base                      582
vertical_front_panel           433
wheel     